# Does the model represent *illegality* as distinct from *harmfulness*?

**What this notebook does, in one paragraph.** We show a small open model 240 short scenarios, arranged as a 2×2: illegal-and-harmful, illegal-but-harmless, legal-but-harmful, legal-and-harmless, four per topic so the topics are matched. We record the model's internal state at the end of each sentence and train the simplest possible detector (a linear probe) to separate illegal from legal on the two *easy* corners (illegal-harmful vs legal-harmless), where legality and harm agree. Then we test it on the two *hard* corners, where they come apart. If the probe still says "illegal" for jaywalking and "legal" for lawful cruelty, the model carries a legality representation that is not just harm. If it does not, a "legal compliance monitor" built on probing is a harm detector with a misleading name. Either way we compare against just asking the model, against a harm probe, and against a bag-of-words classifier on the sentences, and we measure the angle between the legality and harm directions.

**How to use it.** Run top to bottom. Every code cell has a note above it. Settings are in one cell. The clock starts at ⏱; the dataset hand-check is the first clocked task and the one that decides whether the project is real.

**Prior work this must answer** (cite in the write-up): Sadhu et al. 2026 (arXiv 2608.16852) built a compliance readout on Qwen3.5 and found cheap detectors at chance on rule composition; Schwarz 2026 (2607.13075) found harm probes are topic detectors on surface-matched controls. Neither crossed legality with harm on matched topics, measured the angle between the two directions, or intervened. That cross is this project.

### 1 · GPU check
**Runtime → Change runtime type → GPU.** A free T4 is enough; this project is one forward pass per sentence.

In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip() or "NO GPU — Runtime → Change runtime type → GPU")

### 2 · Settings — the only cell you edit
- `MODEL`: Qwen3.5-4B first; 9B if you have an L4/A100 and time.
- `RUN`: name for this session's files.
- `TRAIN` / `TEST`: the quadrants the probe learns from and the quadrants it is tested on. The default is the design: train where legality and harm agree, test where they disagree.
- `DRIVE_DIR`: where results persist.

In [ ]:
MODEL     = "Qwen/Qwen3.5-4B"
RUN       = "lp_4b"
TRAIN     = "illegal_harmful,legal_harmless"
TEST      = "illegal_harmless,legal_harmful"
DRIVE_DIR = "/content/drive/MyDrive/mats12_runs"


### 3 · Get the code and connect Drive (uncounted)
Clones the private repo (needs `GITHUB_TOKEN` in Colab Secrets), installs packages, moves `data/`, `figures/`, `journal/` onto Drive.

In [ ]:
import os, subprocess
from google.colab import drive, userdata
drive.mount("/content/drive")
os.makedirs(DRIVE_DIR, exist_ok=True)
from getpass import getpass
try:
    token = userdata.get("GITHUB_TOKEN")          # Colab Secrets panel (key icon) → GITHUB_TOKEN, "Notebook access" ON
except Exception:
    token = None
if not token:
    print("No GITHUB_TOKEN found in Colab Secrets. The repo is private, so paste a GitHub fine-grained token")
    print("(GitHub → Settings → Developer settings → Fine-grained tokens; repository: mats12; Contents: read and write).")
    token = getpass("GitHub token: ").strip()
url = f"https://{token}@github.com/martinherje/mats12.git"
if not os.path.exists("/content/mats12"):
    r = subprocess.run(["git", "clone", "-q", url, "/content/mats12"], capture_output=True, text=True)
else:
    r = subprocess.run(["git", "-C", "/content/mats12", "pull", "-q"], capture_output=True, text=True)
if r.returncode != 0:
    raise SystemExit("git failed: " + r.stderr.replace(token, "<token>").strip() + "\nCheck the token has access to martinherje/mats12 (Contents: read).")
%cd /content/mats12
!git log --oneline -1
!bash scripts/colab_setup.sh "$DRIVE_DIR" 

### 4 · Load the model once (uncounted)
Expect 32 layers, width 2560, memory well under the card's total.

In [ ]:
!python scripts/gpu_smoke.py --model $MODEL

## ⏱ The clock starts here
Answer `journal/design-questions-legality-probe.md` (Files panel → `mats12/journal`), copy the answers into `journal/highlights.md` with the date, start Toggl.

### 5 · The dataset, and the hand-check that decides whether the project is real
`data/scenarios.csv` holds 240 candidate sentences: 60 topics × 4 quadrants, US law, written by Claude on 9 Sep and **not yet checked by anyone**. This cell prints the counts and a sample. Your job, on the clock: open the CSV in the Files panel (or the table below), read every sentence, and for each one set `hand_checked=1`, fix `legal`/`harmful` where you disagree (and set `relabelled=1`), and set `borderline=1` where a competent lawyer could argue either way. Fix the `quadrant` column to match if you change a label (or let the validator tell you). Save. The write-up reports: how many you read, how many you relabelled, and the disagreement categories. Aim for all 240; if time is short, do the two off-diagonal quadrants first, since the test lives there.

In [ ]:
import pandas as pd
!python scripts/validate_scenarios.py data/scenarios.csv
df = pd.read_csv("data/scenarios.csv")
pd.set_option("display.max_colwidth", 120)
display(df[["id", "quadrant", "topic", "borderline", "text"]].sample(12, random_state=0))
print("\nEdit data/scenarios.csv in the Files panel (double-click), then re-run this cell to validate.")

### 6 · Record the model's internal state for every sentence
One forward pass per sentence; saves the residual stream at the last token, every layer. Look for "acts (240, 33, 2560)". The last token is where a sentence-level judgement has to be available; the design sheet names the alternatives.

In [ ]:
!python scripts/extract_activations.py --model $MODEL --scenarios data/scenarios.csv --run $RUN --batch-size 16

### 7 · The probe, the layer sweep, and the three controls in one go
Trains a logistic-regression probe for *legal* on the easy corners and tests it on the hard corners, at every layer. Same command also runs:
- **shuffled-label control** (what a probe scores when there is nothing to learn),
- **bag-of-words baseline** (what you can get from the words alone: if this matches the probe, the signal is lexical),
- **legality-vs-harm cosine** over the full balanced 2×2 (how far apart the two directions are; near 1 means one axis).

The figure is the headline plot. The JSON records the best layer.

In [ ]:
!python scripts/train_probe.py --run $RUN --label legal --train $TRAIN --test $TEST --contrast harmful --control --bow
import json; from IPython.display import Image, display
best = json.load(open(f"data/processed/probe_{RUN}_legal.json"))
print("best layer:", best["best_layer"], {k: round(v, 3) for k, v in best["best"].items() if isinstance(v, float)})
display(Image(f"figures/probe_{RUN}_legal.png"))

### 8 · Is the legality probe just a harm probe wearing a hat?
Trains a probe for *harmful* on the same easy corners and scores it against the *legality* labels on the hard corners. If this transfers as well as the legality probe did, legality is not separably represented: whatever the probe reads is harm. The predicted-in-advance negative (Shah et al. 2025: harm subconcepts collapse to one axis) is that these two numbers are close.

In [ ]:
!python scripts/train_probe.py --run $RUN --label harmful --eval-label legal --train $TRAIN --test $TEST
import json
h = json.load(open(f"data/processed/probe_{RUN}_harmful_eval-legal.json")); l = json.load(open(f"data/processed/probe_{RUN}_legal.json"))
print(f"legality probe → legality labels on the hard corners: {l['best'].get('test_acc'):.3f} (layer {l['best_layer']})")
print(f"harm probe     → legality labels on the hard corners: {h['best'].get('test_acc'):.3f} (layer {h['best_layer']})")

### 9 · Just ask the model
The cheapest baseline and the one Nanda names first. Shows every sentence to the model and asks "Is this illegal? Yes or No", then "Is this harmful?", and scores the answers, including on the hard corners only. If asking beats the probe, say so; the probe would then be a monitoring convenience, not a discovery. The refusal rate per quadrant is printed too.

In [ ]:
!python scripts/ask_model.py --run $RUN --label legal --model $MODEL
!python scripts/ask_model.py --run $RUN --label harmful --model $MODEL

### 10 · Hand-check the probe's mistakes
Prints the hard-corner sentences the best-layer probe got wrong. Read them: are they the borderline ones? Are they the ones with legality words in them? Write what you see in `journal/verification-log.md`, and recompute the best layer's accuracy yourself from the saved activations (the code below does it from scratch so you can follow it line by line).

In [ ]:
import numpy as np, pandas as pd, json
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
z = np.load(f"data/processed/acts_{RUN}.npz"); acts = z["acts"].astype(np.float32)
cols = {k[4:]: z[k] for k in z.files if k.startswith("col_")}
L = json.load(open(f"data/processed/probe_{RUN}_legal.json"))["best_layer"]
y = cols["legal"].astype(int); quad = cols["quadrant"]
tr = np.isin(quad, TRAIN.split(",")); te = np.isin(quad, TEST.split(","))
clf = make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=2000)).fit(acts[tr, L], y[tr])
pred = clf.predict(acts[te, L])
print(f"recomputed hard-corner accuracy at layer {L}: {(pred == y[te]).mean():.3f}  (should match cell 7)")
wrong = np.where(te)[0][pred != y[te]]
print(f"\n{len(wrong)} mistakes on the hard corners:")
for i in wrong:
    print(f"  [{cols['quadrant'][i]:17s} borderline={cols['borderline'][i]}] {cols['text'][i]}")

### 11 · Optional stretch: does the legality direction do anything?
Steers the model along the legality difference-of-means direction while it answers "Is this illegal?" for the hard-corner sentences, and counts how many answers flip, against a random direction. Turns a correlational probe into an intervention result. Only if the hours allow; the design sheet's stop rule puts this last.

In [ ]:
import sys, json, re, numpy as np, torch, pandas as pd; sys.path.insert(0, "scripts")
from common import load_model, pick_device, pick_dtype
from steer import Intervention, save_directions
z = np.load(f"data/processed/acts_{RUN}.npz"); acts = z["acts"].astype(np.float32); cols = {k[4:]: z[k] for k in z.files if k.startswith("col_")}
L = json.load(open(f"data/processed/probe_{RUN}_legal.json"))["best_layer"]
tr = np.isin(cols["quadrant"], TRAIN.split(","))
save_directions(f"data/processed/direction_{RUN}_legal.npz", acts[tr], cols["legal"][tr].astype(int), "legal")
d = torch.tensor(np.load(f"data/processed/direction_{RUN}_legal.npz")["dirs"][L])
device = pick_device("auto"); tok, model = load_model(MODEL, device, pick_dtype("auto", device)); tok.padding_side = "left"
hard = pd.read_csv("data/scenarios.csv").query("quadrant in @TEST.split(',')")
def ask(texts, hook=None):
    prompts = [tok.apply_chat_template([{"role": "user", "content": f"Under the law of the United States, is the following action illegal? Answer with exactly one word: Yes or No.\n\nAction: {t}"}], tokenize=False, add_generation_prompt=True, enable_thinking=False) for t in texts]
    enc = tok(prompts, return_tensors="pt", padding=True).to(device)
    if hook: hook.__enter__()
    try:
        with torch.no_grad(): out = model.generate(**enc, max_new_tokens=4, do_sample=False, pad_token_id=tok.pad_token_id)
    finally:
        if hook: hook.__exit__(None, None, None)
    return ["yes" if re.match(r"\s*yes", tok.decode(o[enc['input_ids'].shape[1]:], skip_special_tokens=True), re.I) else "no" for o in out]
ALPHA = 8.0   # steering strength in units of the unit direction; sweep 4/8/16 if nothing moves
base = ask(hard.text.tolist())
plus = ask(hard.text.tolist(), Intervention(model, [L], d, mode="add", alpha=ALPHA))     # push toward "legal"
minus = ask(hard.text.tolist(), Intervention(model, [L], d, mode="add", alpha=-ALPHA))   # push toward "illegal"
rand = ask(hard.text.tolist(), Intervention(model, [L], None, mode="random", alpha=ALPHA))
flip = lambda a, b: np.mean([x != y for x, y in zip(a, b)])
print(f"'illegal' answers: base {base.count('yes')}/{len(base)} | +legal dir {plus.count('yes')} | −legal dir {minus.count('yes')} | random {rand.count('yes')}")
print(f"flip rate vs base: +dir {flip(base, plus):.2f}, −dir {flip(base, minus):.2f}, random {flip(base, rand):.2f}")

### 12 · Save your notes to GitHub
Commits `journal/` and the edited `data/scenarios.csv` (your hand-check is part of the record). Results are already on Drive.

In [ ]:
!git config user.email "mherje@live.com" && git config user.name "Martin Herje"
!rm -f journal && cp -r "$DRIVE_DIR/journal" journal && git add journal data/scenarios.csv && (git commit -qm "journal + hand-checked scenarios: Colab session" || true) && git push -q origin main && echo pushed
!rm -rf journal && ln -s "$DRIVE_DIR/journal" journal